<a href="https://colab.research.google.com/github/mikeiasribeiro/engenharia-de-prompt-e-aplica-es-em-IA/blob/main/aula08_automacaoComIa_6f.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Automação com ia - 17/04

nome: Mikeias ribeiro

In [ ]:
import random # Importa a biblioteca random para gerar números aleatórios (agora apenas para referência, pois usaremos input)

# Solicita ao usuário que digite a temperatura atual do sistema.
# Converte a entrada para um número inteiro, pois o input() retorna uma string.
temperatura_atual = int(input("Digite a temperatura atual do sistema em °C: "))

# Exibe a temperatura atual do sistema usando concatenação de strings.
print("Temperatura atual do sistema: " + str(temperatura_atual) + "°C")

# Verifica se a temperatura atual é maior que 30°C.
if temperatura_atual > 30:
    # Se for maior que 30, exibe um alerta de superaquecimento.
    print("ALERTA: Superaquecimento detectado! Temperatura acima de 30°C.")
else:
    # Caso contrário, indica que a temperatura está normal.
    print("Temperatura normal.")


Digite a temperatura atual do sistema em °C: 978
Temperatura atual do sistema: 978°C
ALERTA: Superaquecimento detectado! Temperatura acima de 30°C.


organização de arquivos

In [24]:
import os
import shutil
import logging
from datetime import datetime

# ─────────────────────────────────────────
#  CONFIGURAÇÕES
# ─────────────────────────────────────────
PASTA_ALVO  = "./arquivos_bagunçados"
LOG_FILE    = "organizador.log"
DRY_RUN     = False   # True = só simula, não move nada

CATEGORIAS = {
    # Imagens
    ".jpg": "Imagens", ".jpeg": "Imagens", ".png": "Imagens",
    ".gif": "Imagens", ".bmp": "Imagens",  ".svg": "Imagens",
    # Documentos
    ".pdf": "PDFs", ".docx": "Documentos", ".doc": "Documentos",
    ".txt": "Textos", ".md": "Textos",
    # Planilhas
    ".xlsx": "Planilhas", ".xls": "Planilhas", ".csv": "Planilhas",
    # Vídeos
    ".mp4": "Videos", ".mov": "Videos", ".avi": "Videos", ".mkv": "Videos",
    # Áudios
    ".mp3": "Audios", ".wav": "Audios", ".flac": "Audios",
    # Código
    ".py": "Codigo", ".js": "Codigo", ".html": "Codigo", ".css": "Codigo",
    # Compactados
    ".zip": "Compactados", ".rar": "Compactados", ".tar": "Compactados",
}

# ─────────────────────────────────────────
#  CONFIGURAÇÃO DE LOG
# ─────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ─────────────────────────────────────────
#  FUNÇÕES AUXILIARES
# ─────────────────────────────────────────
def resolver_conflito(destino: str) -> str:
    """Se o arquivo já existe no destino, adiciona um sufixo numérico."""
    if not os.path.exists(destino):
        return destino
    base, ext = os.path.splitext(destino)
    contador = 1
    while os.path.exists(f"{base}_{contador}{ext}"):
        contador += 1
    return f"{base}_{contador}{ext}"


def criar_pasta(caminho: str) -> None:
    """Cria a pasta se não existir (respeita DRY_RUN)."""
    if not os.path.exists(caminho):
        if not DRY_RUN:
            os.makedirs(caminho)
        log.info(f"{'[SIMULAÇÃO] ' if DRY_RUN else ''}Pasta criada: {caminho}")


def mover_arquivo(origem: str, destino: str) -> None:
    """Move o arquivo tratando conflitos de nome."""
    destino = resolver_conflito(destino)
    if not DRY_RUN:
        shutil.move(origem, destino)
    log.info(f"{'[SIMULAÇÃO] ' if DRY_RUN else ''}Movido: {os.path.basename(origem)}  →  {os.path.dirname(destino)}/")


# ─────────────────────────────────────────
#  FUNÇÃO PRINCIPAL
# ─────────────────────────────────────────
def organizar(pasta: str) -> dict:
    """Organiza os arquivos e retorna um dicionário com estatísticas."""
    if not os.path.exists(pasta):
        log.error(f"Pasta não encontrada: '{pasta}'")
        return {}

    stats = {"movidos": 0, "ignorados": 0, "erros": 0, "categorias": {}}
    inicio = datetime.now()

    log.info("─" * 50)
    log.info(f"Iniciando organização em: {pasta}")
    if DRY_RUN:
        log.info("MODO SIMULAÇÃO ATIVO — nenhum arquivo será movido.")
    log.info("─" * 50)

    for nome in sorted(os.listdir(pasta)):
        caminho_origem = os.path.join(pasta, nome)

        # Pula subpastas
        if os.path.isdir(caminho_origem):
            stats["ignorados"] += 1
            continue

        _, ext = os.path.splitext(nome)
        ext = ext.lower()

        categoria = CATEGORIAS.get(ext, "Outros")
        pasta_destino = os.path.join(pasta, categoria)

        try:
            criar_pasta(pasta_destino)
            mover_arquivo(caminho_origem, os.path.join(pasta_destino, nome))
            stats["movidos"] += 1
            stats["categorias"][categoria] = stats["categorias"].get(categoria, 0) + 1

        except PermissionError:
            log.error(f"Sem permissão para mover: {nome}")
            stats["erros"] += 1
        except Exception as e:
            log.error(f"Erro inesperado com '{nome}': {e}")
            stats["erros"] += 1

    duracao = (datetime.now() - inicio).total_seconds()
    return {**stats, "duracao_s": round(duracao, 3)}


def exibir_relatorio(stats: dict) -> None:
    """Exibe um resumo formatado no terminal."""
    if not stats:
        return

    print("\n" + "=" * 50)
    print("  RELATORIO FINAL")
    print("=" * 50)
    print(f"  Arquivos movidos : {stats['movidos']}")
    print(f"  Ignorados        : {stats['ignorados']}")
    print(f"  Erros            : {stats['erros']}")
    print(f"  Tempo total      : {stats['duracao_s']}s")

    if stats["categorias"]:
        print("\n  Por categoria:")
        for cat, qtd in sorted(stats["categorias"].items()):
            print(f"    {cat:<15} {qtd} arquivo(s)")

    print("=" * 50)
    print(f"  Log salvo em: {LOG_FILE}\n")


# ─────────────────────────────────────────
#  EXECUÇÃO
# ─────────────────────────────────────────
if __name__ == "__main__":
    resultado = organizar(PASTA_ALVO)
    exibir_relatorio(resultado)

ERROR:__main__:Pasta não encontrada: './arquivos_bagunçados'


Consulta de CEP

In [25]:
import requests # Importa a biblioteca requests para fazer requisições HTTP a APIs externas.

def consultar_cep(cep: str) -> dict:
    """
    Consulta um CEP utilizando a API ViaCEP.

    Args:
        cep (str): O número do CEP a ser consultado (pode conter caracteres não numéricos).

    Returns:
        dict: Um dicionário contendo os dados do endereço ou uma mensagem de erro.
    """
    # 1. Limpeza do CEP: Remove todos os caracteres não numéricos do CEP.
    # Isso garante que apenas os dígitos essenciais sejam usados na consulta.
    cep_limpo = ''.join(filter(str.isdigit, cep))

    # 2. Validação do CEP: Verifica se o CEP limpo tem exatamente 8 dígitos.
    # APIs de CEP geralmente exigem 8 dígitos.
    if len(cep_limpo) != 8:
        return {"erro": "CEP inválido. Deve conter 8 dígitos numéricos."}

    # 3. Construção da URL da API: Monta a URL para a requisição à ViaCEP.
    # A ViaCEP oferece um endpoint RESTful simples para consulta de CEPs.
    url = f"https://viacep.com.br/ws/{cep_limpo}/json/"

    try:
        # 4. Requisição HTTP: Envia uma requisição GET para a API da ViaCEP.
        response = requests.get(url)

        # 5. Verificação de Status da Resposta: Levanta um erro HTTP para status 4xx/5xx.
        # Isso ajuda a identificar problemas de conexão ou CEPs inexistentes na API.
        response.raise_for_status()

        # 6. Conversão da Resposta para JSON: Analisa o corpo da resposta como JSON.
        data = response.json()

        # 7. Verificação de Erro da API: A ViaCEP retorna {'erro': true} para CEPs não encontrados.
        if "erro" in data and data["erro"]:
            return {"erro": "CEP não encontrado na base de dados."}
        else:
            # 8. Retorno dos Dados: Se tudo ocorrer bem, retorna os dados do endereço.
            return data

    except requests.exceptions.RequestException as e:
        # Captura erros relacionados à rede (ex: sem internet, DNS não encontrado).
        return {"erro": f"Erro de conexão à API ViaCEP: {e}"}
    except ValueError:
        # Captura erros se a resposta não puder ser decodificada como JSON (resposta malformada).
        return {"erro": "Erro ao decodificar a resposta JSON da API."}


# --- Bloco de Execução Principal do Script ---
# Este bloco é executado apenas quando o script é rodado diretamente.
if __name__ == "__main__":
    # Solicita ao usuário que digite o CEP.
    cep_digitado = input("Digite o CEP (apenas números, ou com pontos/hífens): ")

    # Chama a função de consulta de CEP.
    resultado = consultar_cep(cep_digitado)

    # Verifica se houve erro na consulta e exibe a mensagem apropriada.
    if "erro" in resultado:
        print(f"Erro: {resultado['erro']}")
    else:
        # Se não houver erro, exibe os detalhes do endereço formatados.
        print("\n--- Detalhes do Endereço ---")
        print(f"CEP: {resultado.get('cep', 'N/A')}")
        print(f"Logradouro: {resultado.get('logradouro', 'N/A')}")
        print(f"Complemento: {resultado.get('complemento', 'N/A')}")
        print(f"Bairro: {resultado.get('bairro', 'N/A')}")
        print(f"Cidade: {resultado.get('localidade', 'N/A')}")
        print(f"Estado: {resultado.get('uf', 'N/A')}")
        print(f"DDD: {resultado.get('ddd', 'N/A')}")
        print(f"IBGE: {resultado.get('ibge', 'N/A')}")
        print(f"GIA: {resultado.get('gia', 'N/A')}")

Digite o CEP (apenas números): 70390045

--- Detalhes do Endereço ---
CEP: 70390-045
Logradouro: Quadra SEPS 704/904
Complemento: 
Bairro: Asa Sul
Cidade: Brasília
Estado: DF
DDD: 61




---

Sistema de notificação

In [26]:
# ═══════════════════════════════════════════════════════
#  MISSÃO 03 — SISTEMA DE NOTIFICAÇÕES
#  Disciplina: Engenharia de Prompt e Aplicações em IA
#
#  Objetivo: Disparar alertas automáticos (console e
#  e-mail simulado) quando uma condição crítica for
#  atingida em métricas do sistema.
# ═══════════════════════════════════════════════════════

# Biblioteca para gerar valores aleatórios (simula leitura real)
import random

# Biblioteca para capturar data e hora atual
from datetime import datetime

# Biblioteca para contagem de elementos
import collections


# ─────────────────────────────────────────
#  LIMITES DE ALERTA POR MÉTRICA
#
#  Cada métrica tem dois limiares:
#    - "warning"  → sinal de atenção (amarelo)
#    - "critical" → situação grave   (vermelho)
# ─────────────────────────────────────────
LIMITES = {
    "cpu":     {"warning": 70, "critical": 90},
    "memoria": {"warning": 75, "critical": 85},
    "disco":   {"warning": 80, "critical": 90},
    "temp":    {"warning": 65, "critical": 80},
}


# ─────────────────────────────────────────
#  COLETA DE DADOS
# ─────────────────────────────────────────
def coletar_dados() -> dict:
    """
    Simula a leitura das métricas do sistema.
    Em produção, aqui entrariam bibliotecas como
    psutil para leitura real de CPU, memória etc.
    """
    return {
        "cpu":     random.randint(0, 100),   # % de uso do processador
        "memoria": random.randint(0, 100),   # % de uso da RAM
        "disco":   random.randint(0, 100),   # % de uso do armazenamento
        "temp":    random.randint(30, 95),   # temperatura em graus Celsius
    }


# ─────────────────────────────────────────
#  AVALIAÇÃO DE NÍVEL
# ─────────────────────────────────────────
def avaliar_nivel(valor: float, limites: dict) -> str:
    """
    Compara o valor da métrica com os limites definidos
    e retorna o nível de severidade correspondente.

    Retorna:
        "CRITICAL" → valor acima do limite crítico
        "WARNING"  → valor acima do limite de aviso
        "OK"       → valor dentro do normal
    """
    if valor >= limites["critical"]:
        return "CRITICAL"
    elif valor >= limites["warning"]:
        return "WARNING"
    return "OK"


# ─────────────────────────────────────────
#  NOTIFICAÇÃO NO CONSOLE
# ─────────────────────────────────────────
def enviar_alerta_console(metrica: str, valor: float, nivel: str) -> None:
    """
    Exibe uma mensagem de alerta no terminal.
    O ícone muda conforme o nível:
      🔴 = CRITICAL  |  🟡 = WARNING
    """
    hora  = datetime.now().strftime("%H:%M:%S")           # horário do alerta
    icone = "🔴" if nivel == "CRITICAL" else "🟡"         # ícone por nível

    # Formato: [hora] [NÍVEL] MÉTRICA em XX%
    print(f"  {icone} [{hora}] [{nivel}] {metrica.upper()} em {valor}%")


# ─────────────────────────────────────────
#  E-MAIL SIMULADO
# ─────────────────────────────────────────
def simular_email(metrica: str, valor: float, nivel: str) -> None:
    """
    Simula o envio de um e-mail de emergência.
    Em produção, este bloco seria substituído por
    uma chamada real via smtplib ou API de e-mail.
    Só é chamada em situações CRITICAL.
    """
    # Descobre qual limite foi ultrapassado para exibir no corpo
    limite_atingido = LIMITES[metrica]["critical" if nivel == "CRITICAL" else "warning"]

    print(f"\n  {'─' * 44}")
    print(f"  📧  E-MAIL SIMULADO")
    print(f"  Para    : admin@sistema.com")
    print(f"  Assunto : [{nivel}] Alerta — {metrica.upper()} em {valor}%")
    print(f"  Corpo   : A métrica '{metrica}' atingiu {valor}%,\n            acima do limite {nivel.lower()} de {limite_atingido}%")
    print(f"  {'─' * 44}\n")


# ─────────────────────────────────────────
#  VERIFICAÇÃO INDIVIDUAL DE MÉTRICA
# ─────────────────────────────────────────
def verificar_metrica(metrica: str, valor: float) -> str:
    """
    Orquestra a verificação de uma única métrica:
      1. Avalia o nível (OK / WARNING / CRITICAL)
      2. Dispara alerta no console se necessário
      3. Dispara e-mail simulado apenas se CRITICAL
    Retorna o nível encontrado.
    """
    # Busca os limites da métrica; ignora se não estiver mapeada
    limites = LIMITES.get(metrica)
    if limites is None:
        return "OK"

    # Determina o nível de severidade
    nivel = avaliar_nivel(valor, limites)

    # WARNING → apenas alerta no console
    if nivel == "WARNING":
        enviar_alerta_console(metrica, valor, nivel)

    # CRITICAL → alerta no console + e-mail simulado
    elif nivel == "CRITICAL":
        enviar_alerta_console(metrica, valor, nivel)
        simular_email(metrica, valor, nivel)

    # OK → nenhuma notificação necessária
    return nivel


# ─────────────────────────────────────────
#  RELATÓRIO FINAL
# ─────────────────────────────────────────
def exibir_relatorio(dados: dict, resultados: dict) -> None:
    """
    Imprime um painel com o status de todas as métricas
    e um resumo de quantas estão em cada nível.
    """
    print("\n" + "═" * 46)
    print("  RELATÓRIO DO SISTEMA")
    print("═" * 46)

    # Itera sobre cada métrica e exibe seu status com ícone
    for metrica, valor in dados.items():
        nivel = resultados.get(metrica, "OK")

        # Escolhe ícone de acordo com o nível
        if nivel == "CRITICAL":
            icone = "🔴"
        elif nivel == "WARNING":
            icone = "🟡"
        else:
            icone = "🟢"

        # Linha formatada: ícone + nome + valor + nível
        print(f"  {icone}  {metrica.upper():<10} {valor:>4}%   [{nivel}]")

    print("═" * 46)

    # Contagem por nível para o resumo final usando Counter para concisão
    contagem_niveis = collections.Counter(resultados.values())
    criticos = contagem_niveis["CRITICAL"]
    warnings = contagem_niveis["WARNING"]
    ok       = contagem_niveis["OK"]

    print(f"  Críticos : {criticos}  |  Avisos : {warnings}  |  OK : {ok}")
    print("═" * 46 + "\n")


# ─────────────────────────────────────────
#  FUNÇÃO PRINCIPAL
# ─────────────────────────────────────────
def monitorar_sistema() -> None:
    """
    Ponto de entrada principal do monitor.
    Fluxo:
      1. Exibe cabeçalho com data/hora
      2. Coleta as métricas do sistema
      3. Verifica cada métrica individualmente
      4. Exibe relatório consolidado
    """
    # Cabeçalho com timestamp
    print("\n" + "═" * 46)
    print("  🖥️   MONITOR DE SISTEMA INICIADO")
    print(f"  {datetime.now().strftime('%d/%m/%Y  %H:%M:%S')}")
    print("═" * 46)

    # Coleta os dados simulados
    dados = coletar_dados()
    print("\n  Verificando métricas...\n")

    # Percorre cada métrica e armazena o nível retornado
    resultados = {}
    for metrica, valor in dados.items():
        resultados[metrica] = verificar_metrica(metrica, valor)

    # Exibe o painel final com todos os status
    exibir_relatorio(dados, resultados)


# ─────────────────────────────────────────
#  PONTO DE ENTRADA
#  Garante que o código só executa quando
#  rodado diretamente (não ao ser importado)
# ─────────────────────────────────────────
if __name__ == "__main__":
    monitorar_sistema()


══════════════════════════════════════════════
  🖥️   MONITOR DE SISTEMA INICIADO
  18/04/2026  00:52:23
══════════════════════════════════════════════

  Verificando métricas...

  🔴 [00:52:23] [CRITICAL] TEMP em 86%

  ────────────────────────────────────────────
  📧  E-MAIL SIMULADO
  Para    : admin@sistema.com
  Assunto : [CRITICAL] Alerta — TEMP em 86%
  Corpo   : A métrica 'temp' atingiu 86%,
            acima do limite critical de 80%.
  ────────────────────────────────────────────


══════════════════════════════════════════════
  RELATÓRIO DO SISTEMA
══════════════════════════════════════════════
  🟢  CPU           8%   [OK]
  🟢  MEMORIA      66%   [OK]
  🟢  DISCO        77%   [OK]
  🔴  TEMP         86%   [CRITICAL]
══════════════════════════════════════════════
  Críticos : 1  |  Avisos : 0  |  OK : 3
══════════════════════════════════════════════

